# Langchain

Langchain is a framework for developing applications powered by language models

### OverView:
* Installation
* LLM's
* Prompt Templates
* Chains
* Agents and Tools
* Memory
* Document Loaders
* Indexes

## **01: Installation**

In [ ]:
!pip install langchain langchain_community --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## **02: Setup Environment**

In [ ]:
import os

In [ ]:
os.environ['OPENAI_API_KEY'] = "Your_openai_api_key"
os.environ['HUGGINGFACEHUB_API_TOKEN'] = "Token_here"

## **03: Large Language Models**

The basic building block of Langchain is a Large Language Model which takes text as input and generates more text

Temperature value -> how creative we want our model to be
0 -> temperature means model is very safe, it is not taking any risks
1 -> it will take risks and will hallucinate, meaning it will generate random data.

## OpenAI

### Example 1

In [ ]:
!pip install openai --quiet

In [ ]:
from langchain.llms import OpenAI
llm = OpenAI(temperature = 0.9)

In [ ]:
text = "what would be a good company name for a company that makes baggy jeans?"

In [ ]:
print(llm.predict(text))

In [ ]:
print(llm.invoke(text))

### Example 2

In [ ]:
name = llm.predict("I want to open a Tech company that focuses on Cybersecurity. Suggest me names.")
print(name)

NameError: name 'llm' is not defined

In [ ]:
response.llm("I want to open a Tech company that focuses on Cybersecurity. Suggest me names.")
print(response)

### Hugging Face

### Example 1

In [ ]:
!pip install huggingface_hub --quiet

In [ ]:
from langchain import HuggingFaceHub

In [ ]:
llm = HuggingFaceHub(repo_id = "google/flan-t5-large",
                     model_kwargs = {"temperature": 0, "max_length": 64})
llm("translate English to German: How old are you?")

### Example 2

In [ ]:
llm = HuggingFaceHub(repo_id = "google/flan-t5-large",
                     model_kwargs = {"temperature": 0, "max_length": 64})
name = llm.predict("I want to open a Tech company that focuses on Cybersecurity. Suggest me names.")
print(name)

## **04: Prompt Templates**

## Example 1

In [ ]:
from langchain.prompts import PromptTemplate

prompt_template_name = PromptTemplate(
    input_variables = ['cuisine'],
    template = "I want to open a restaurent for {cuisine} food. Suggest a fancy name for this."
)
p = prompt_template_name.format(cusine = "Italian")
print(p)

## Example 2

In [ ]:
from langchain.prompts import PromptTemplate

prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}")
prompt.formet(product = "colorful socks")

## **05: Chains**

Comnine LLM's and Prompts in multi-step workflows

Now as we have **model:**
llm = OpenAI(temperature = 0.9)
and the **Prompt Template:**
prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}")
prompt.format(product = "colorful socks")

Now using Chains, we'll link together model and tht PromptTemplate and the other Chains

The simplest and most common type of Chain is LLMChain, which passe the input first to Prompt Template and then to a Large Language Model

## Example 1

In [ ]:
prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}")
prompt.format(product = "colorful socks")

In [ ]:
from langchain.chains import LLMChain
chain = LLMChain(llm = llm, prompt = prompt)
response = chain.run("colorful socks")
print(response)

## Example 2

In [ ]:
prompt_template_name = PromptTemplate(
    input_variables = ["cuisine"],
    template = "I want to open a restaurent for {cuisine} food. Suggest a fancy name for this."
)

In [ ]:
chain = LLMChain(llm = llm, prompt = prompt)
response = chain.run("Chinese")
print(response)

In [ ]:
chain = LLMChain(llm = llm, prompt = prompt_template_name, verbose = True)
response = chian.run("Chinese")
print(response)

Can we combine Multile PromptTemplates, we will try to combine Multiple PromptTemplates

The Output from the first PromptTemplate is passed to the next PromptTemplate as input

### To combine the chain and to set a sequence for that we use SimpleSequentialChain

### Simple Sequential Chain

In [ ]:
llm = OpenAI(temperatue = 0.9)
prompt_template_name = PromptTemplate(
    input_variables = ["cuisine"],
    template = "I want to open a restaurent for {cuisine} food. Suggest a fancy name for this."
)
name_chain = LLMChain(llm = llm, prompt = prompt_template_name)

prompt_template_items = PromptTemplate(
    input_variables = ["restaurent_name"],
    template = """Suggest some menu items for {restaurent_name}"""
)

food_item_chain = LLMChain(llm = llm, prompt = prompt_template_items)

In [ ]:
from langchain.chains import SimpleSequentialChain

In [ ]:
chain = SimpleSequentialChain(chains = [name_chain, food_item_chain])
content = chain.run("indian")
print(content)

There is a issue with SimpleSequentualChain as it only shows last input information

### To show entire information, we'll need to use SequentialChain

## Sequential Chain

In [ ]:
llm = OpenAI(temperature = 0.7)
prompt_template_name = PromptTemplate(
    input_variables = ["cuisine"],
    template = "I want to open a restaurent for {cuisine} food. Suggest a fancy name for this."
)
name_chain = LLMChain(llm = llm, prompt = prompt_template_name, output_key = "restaurent_name")

In [ ]:
prompt_template_items = PromptTemplate(
    input_variables = ["restaurent_name"],
    template = "Suggest some meny Items for {restaurent_name}"
)
food_item_chain = LLMChain(llm = llm, prompt = prompt_template_items, output_key = "menu_items")

In [ ]:
from langchain.chains import SequentialChain

In [ ]:
chain = SequentialChain(
    chains = [name_chain, food_item_chain],
    input_variables = ["cuisine"],
    output_variables = ["restaurent_name", "menu_items"]
)
print(chain({"cuisine": "indian"}))

## **06: Agent and Tools**

Agents involve an LLM making decision about which Actions to take, taking that Action, seeing Observation and repeating that until done.
When used correctly, agents can be extremely powerful. In order to load agents, we should understant the following concepts:
* Tool: A function that perform a specific duty. This can be things like: Google Search, database lookup, Python REPL, other chains.
* LLM: The language model powering the agent
* Agent: The agent to use.

Agent is a very powerful concept in Langchain

For example, i have to travel from Dubai to Canada, I type this to ChatGPT:
-> Give me two flights option form Dubai to Canada on Septemper 1, 2026. ChatGpt will nto be able to answer because it does not have that knowledge yet.

ChatGPT plus has Expedia Plugin, if we enable this plugin it will got to Expedia Plugin and will try to pull information about the Flight i need.

SerpApi is a real-time API to access Google Search Resutlts

### Wikipedia and llm-math tool

In [ ]:
!pip install wikipedia --quiet

  Preparing metadata (setup.py) ... done


In [ ]:
from langchain.agents import AgentType, intialize_agent, load_tools
from langchain.llms import OpenAI

In [ ]:
llm = OpenAI(temperature = 0)

In [ ]:
# The tools we'll give the agent access to. Note that the 'llm-math' tools uses an LLM, so we need to pass that as a parameter
tools = load_tools(["wikipedia", "llm-math"], llm = llm)

# Now we can initialize the agent with the tools, the language model and the type of agent we want to use
agent = intialize_agent(
    tools,
    llm,
    agent = AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose = True
)

agent.run("What was the GPD of US in 2024?")

### **07: Memory**

LLM's like ChatGPT can remember previous conversations.

In [ ]:
llm = OpenAI(temperature = 0.9)

In [ ]:
from langchain import PromptTemplate

prompt_template_name = PromptTemplate(
    input_variables = ['cuisine'],
    template = "I want to open a restaurent for {cuisine} food. Suggest a fancy name for this."
)

In [ ]:
from langchain.chains import LLMChain

chain = LLMChain(llm = llm, prompt = prompt_template_name)
content = chain.run("Chinese")
print(content)

In [ ]:
name = chain.run("Indian")
print(name)

In [ ]:
chain.memory

In [ ]:
# It does not save any previous memory
type(chain.memory)

### **ConversationBufferMemory**

The problem with this memory method is that it will store all the information, this can work for small chatbots or llm's but with Generative AI, this can cause a lot of Memory Usage

In [ ]:
from lanchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory()

chain = LLMChain(llm = llm, prompt = prompt_template_name, memory = memory)
name = chain.run("Mexican")
print(name)

In [ ]:
name = chain.run("Arabic")
print(name)

In [ ]:
print(chain.memory.buffer)

## **ConversationChain**

Now if i want to store only 5 or 10 or 20 conversations then we can use ConversationChain

In [ ]:
from langchain.chains import ConversationChain

convo = ConversationChain(llm = OpenAI(temperature = 0.7))
print(convo.prompt.template)

In [ ]:
convo.run("Who won the first cricket world cup?")

In [ ]:
convo.run("How much is 5 * 5?")

In [ ]:
# ChatGPT will retain memory from previous conversation and answer accordingly
convo.run("Who was the captain of the winning team?")

In [ ]:
print(convo.memory.buffer)

## **ConversationBufferWindowMemory**

In [ ]:
from langchain.chains import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k = 3)
convo = ConversationChain(
    llm = OpenAI(temperature = 0.7),
    memory = memory
)

convo.run("Who won the first cricket world cup?")

In [ ]:
convo.run("How much is 5 * 5?")

In [ ]:
# ChatGPT will retain memory from previous conversation and answer accordingly
convo.run("Who was the captain of the winning team?")

In [ ]:
print(convo.memory.buffer)

## **08: Document Loaders**

In [ ]:
from langchain.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/my_paper.pdf")
pages = loader.load()
pages